# Exploración de StatsBomb Open Data: El Estándar de Oro en Datos de Fútbol
Este notebook sirve como introducción a la librería `statsbombpy` y demuestra la profundidad y granularidad de los datos gratuitos ofrecidos por StatsBomb.

### Configuración del entorno
Importamos las librerías necesarias y configuramos el despliegue de Pandas para visualizar correctamente la gran cantidad de columnas que ofrece el dataset.

In [13]:
# Imports principales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsbombpy import sb
import warnings
warnings.filterwarnings('ignore')

# Configuración de display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

print("Setup completo. Librerías cargadas.")

Setup completo. Librerías cargadas.


### Exploración del catálogo de StatsBomb
Accedemos a la API para listar todas las competiciones disponibles de forma gratuita. Esto nos permite identificar los `competition_id` y `season_id` necesarios para extraer partidos específicos.

In [14]:
# Ver todas las competiciones que ofrece StatsBomb gratis
competitions = sb.competitions()
print(f"Total de competiciones disponibles: {len(competitions)}")
competitions.head(20)

Total de competiciones disponibles: 75


,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-07-06T04:26:07.636270,2025-07-06T04:26:07.636270,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,NaN,NaN,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2024-09-28T01:57:35.846538,NaN,NaN,2024-09-28T01:57:35.846538
3,16,4,Europe,Champions League,male,False,False,2018/2019,2025-05-08T15:10:50.835274,2021-06-13T16:17:31.694,NaN,2025-05-08T15:10:50.835274
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,NaN,2024-02-13T02:35:28.134882
5,16,2,Europe,Champions League,male,False,False,2016/2017,2024-02-13T02:37:32.205154,2021-06-13T16:17:31.694,NaN,2024-02-13T02:37:32.205154
6,16,27,Europe,Champions League,male,False,False,2015/2016,2024-06-12T07:45:38.786894,2021-06-13T16:17:31.694,NaN,2024-06-12T07:45:38.786894
7,16,26,Europe,Champions League,male,False,False,2014/2015,2024-02-12T12:49:54.914228,2021-06-13T16:17:31.694,NaN,2024-02-12T12:49:54.914228
8,16,25,Europe,Champions League,male,False,False,2013/2014,2024-02-12T12:48:48.479157,2021-06-13T16:17:31.694,NaN,2024-02-12T12:48:48.479157
9,16,24,Europe,Champions League,male,False,False,2012/2013,2024-02-12T12:47:34.340413,2021-06-13T16:17:31.694,NaN,2024-02-12T12:47:34.340413


In [15]:
# Mostrar las columnas clave de todas las competiciones
competitions[['competition_id', 'season_id', 'competition_name', 'season_name', 'country_name']].sort_values('competition_name')

,competition_id,season_id,competition_name,season_name,country_name
0,9,281,1. Bundesliga,2023/2024,Germany
1,9,27,1. Bundesliga,2015/2016,Germany
2,1267,107,African Cup of Nations,2023,Africa
20,16,276,Champions League,1970/1971,Europe
19,16,71,Champions League,1971/1972,Europe
...,...,...,...,...,...
70,35,75,UEFA Europa League,1988/1989,Europe
71,53,315,UEFA Women's Euro,2025,Europe
72,53,106,UEFA Women's Euro,2022,Europe
73,72,107,Women's World Cup,2023,International


### Filtrado de partidos por competición
En este caso, filtramos el catálogo para obtener todos los encuentros correspondientes a la Copa del Mundo Qatar 2022.

In [16]:
# Cargar partidos del Mundial Qatar 2022
matches = sb.matches(competition_id=43, season_id=106)
print(f"Total de partidos en Mundial 2022: {len(matches)}")
matches[['match_date', 'home_team', 'away_team', 'home_score', 'away_score', 'competition_stage']].head(15)

Total de partidos en Mundial 2022: 64


,match_date,home_team,away_team,home_score,away_score,competition_stage
0,2022-12-02,Serbia,Switzerland,2,3,Group Stage
1,2022-12-03,Argentina,Australia,2,1,Round of 16
2,2022-11-30,Australia,Denmark,1,0,Group Stage
3,2022-11-24,Brazil,Serbia,2,0,Group Stage
4,2022-11-26,Tunisia,Australia,0,1,Group Stage
5,2022-11-29,Ecuador,Senegal,1,2,Group Stage
6,2022-12-09,Netherlands,Argentina,2,2,Quarter-finals
7,2022-11-24,Uruguay,South Korea,0,0,Group Stage
8,2022-12-10,Morocco,Portugal,1,0,Quarter-finals
9,2022-12-18,Argentina,France,3,3,Final


### Identificación de un partido específico
Buscamos el `match_id` de la final entre Argentina y Francia para profundizar en sus datos de eventos.

In [17]:
# Buscar la final
final = matches[
    (matches['home_team'].isin(['Argentina', 'France'])) & 
    (matches['away_team'].isin(['Argentina', 'France']))
]
print(final[['match_date', 'home_team', 'away_team', 'home_score', 'away_score']])

# Guardar el match_id
final_id = final['match_id'].iloc[0]
print(f"\nMatch ID de la final: {final_id}")

   match_date  home_team away_team  home_score  away_score
9  2022-12-18  Argentina    France           3           3

Match ID de la final: 3869685


### Extracción de la "Metadata" de Eventos
Cargamos todos los eventos del partido. StatsBomb ofrece una granularidad impresionante, superando las 90 columnas de información técnica por cada registro.

In [18]:
# Cargar TODOS los eventos del partido
events = sb.events(match_id=final_id)
print(f"Total de eventos en el partido: {len(events)}")
print(f"Columnas disponibles: {len(events.columns)}")
print("\nPrimeras columnas:")
print(events.columns.tolist()[:30])

Total de eventos en el partido: 4407
Columnas disponibles: 94

Primeras columnas:
['50_50', 'bad_behaviour_card', 'ball_receipt_outcome', 'ball_recovery_offensive', 'ball_recovery_recovery_failure', 'block_deflection', 'block_offensive', 'carry_end_location', 'clearance_aerial_won', 'clearance_body_part', 'clearance_head', 'clearance_left_foot', 'clearance_other', 'clearance_right_foot', 'counterpress', 'dribble_nutmeg', 'dribble_outcome', 'dribble_overrun', 'duel_outcome', 'duel_type', 'duration', 'foul_committed_advantage', 'foul_committed_card', 'foul_committed_offensive', 'foul_committed_penalty', 'foul_committed_type', 'foul_won_advantage', 'foul_won_defensive', 'foul_won_penalty', 'goalkeeper_body_part']


### Análisis de tipos de eventos
Exploramos la frecuencia de cada tipo de evento (pases, tiros, presiones, etc.) para entender el volumen de datos generado en un solo encuentro de élite.

In [19]:
# ¿Qué tipos de eventos hay?
print("Distribución de tipos de eventos:")
print(events['type'].value_counts())

Distribución de tipos de eventos:
type
Pass               1263
Ball Receipt*      1114
Carry               940
Pressure            361
Ball Recovery       115
Duel                 98
Dribble              54
Block                50
Foul Committed       48
Clearance            45
Foul Won             44
Goal Keeper          44
Shot                 38
Miscontrol           35
Dispossessed         34
Dribbled Past        31
Interception         28
Substitution         13
Half Start           10
Half End             10
Injury Stoppage       9
50/50                 8
Tactical Shift        7
Starting XI           2
Bad Behaviour         2
Player Off            1
Player On             1
Offside               1
Shield                1
Name: count, dtype: int64


In [20]:
# Ver los primeros 20 eventos
events[['minute', 'second', 'team', 'player', 'type', 'location']].head(20)

,minute,second,team,player,type,location
0,0,0,Argentina,NaN,Starting XI,NaN
1,0,0,France,NaN,Starting XI,NaN
2,0,0,France,NaN,Half Start,NaN
3,0,0,Argentina,NaN,Half Start,NaN
4,45,0,France,NaN,Half Start,NaN
5,45,0,Argentina,NaN,Half Start,NaN
6,90,0,France,NaN,Half Start,NaN
7,90,0,Argentina,NaN,Half Start,NaN
8,105,0,France,NaN,Half Start,NaN
9,105,0,Argentina,NaN,Half Start,NaN


### Análisis de Goles y Calidad de Finalización (xG)
Filtramos los tiros que terminaron en gol y analizamos el **Expected Goals (xG)**, una métrica avanzada que calcula la probabilidad de éxito de un disparo según su contexto.

In [21]:
# Tomar el primer gol del partido (Messi de penal)
goles = events[events['type'] == 'Shot']
goles_marcados = goles[goles['shot_outcome'] == 'Goal']
print(f"Total de goles en el partido: {len(goles_marcados)}")

# Ver el primer gol con todos sus detalles
primer_gol = goles_marcados.iloc[0]
print(f"\nMinuto {primer_gol['minute']}: {primer_gol['player']} ({primer_gol['team']})")
print(f"Tipo de jugada: {primer_gol['shot_type']}")
print(f"Parte del cuerpo: {primer_gol['shot_body_part']}")
print(f"xG: {primer_gol['shot_statsbomb_xg']:.3f}")
print(f"Ubicación: {primer_gol['location']}")

Total de goles en el partido: 12

Minuto 22: Lionel Andrés Messi Cuccittini (Argentina)
Tipo de jugada: Penalty
Parte del cuerpo: Left Foot
xG: 0.783
Ubicación: [108.0, 40.0]


### Estadísticas comparativas por equipo
Calculamos la distribución de eventos y el volumen de pases para comparar el dominio territorial y el estilo de juego de ambos finalistas.

In [22]:
# Calcular posesión por equipo (basado en cantidad de eventos)
posesion = events.groupby('team').size()
posesion_pct = (posesion / posesion.sum() * 100).round(1)
print("Distribución de eventos por equipo:")
print(posesion_pct)

# Pases por equipo
pases = events[events['type'] == 'Pass']
pases_por_equipo = pases.groupby('team').size()
print(f"\nTotal de pases por equipo:")
print(pases_por_equipo)

Distribución de eventos por equipo:
team
Argentina    53.8
France       46.2
dtype: float64

Total de pases por equipo:
team
Argentina    693
France       570
dtype: int64


## Conclusión
Este breve recorrido demuestra que **StatsBomb Open Data** no solo entrega resultados, sino el contexto completo de lo que sucede en el campo. Con ~4,000 eventos por partido, las posibilidades de análisis táctico y estadístico son prácticamente ilimitadas.